In [1]:
# чтение данных из базы
import pandas as pd
import os, psycopg
from dotenv import load_dotenv
load_dotenv()

TABLE_NAME = "users_churn"# таблица с данными


connection = {"sslmode": "require", "target_session_attrs": "read-write"}
postgres_credentials = {
    "host": os.getenv("DB_DESTINATION_HOST"),
    "port": os.getenv("DB_DESTINATION_PORT"),
    "dbname": os.getenv("DB_DESTINATION_NAME"),
    "user": os.getenv("DB_DESTINATION_USER"),
    "password": os.getenv("DB_DESTINATION_PASSWORD"),
}

connection.update(postgres_credentials)

with psycopg.connect(**connection) as conn:

    with conn.cursor() as cur:
        cur.execute(f"SELECT * FROM {TABLE_NAME}")
        data = cur.fetchall()
        columns = [col[0] for col in cur.description]

df = pd.DataFrame(data, columns=columns)

df[:2]

,id,customer_id,begin_date,end_date,type,paperless_billing,payment_method,monthly_charges,total_charges,internet_service,...,device_protection,tech_support,streaming_tv,streaming_movies,gender,senior_citizen,partner,dependents,multiple_lines,target
0,2043,7361-YPXFS,2017-10-01,NaT,Month-to-month,No,Bank transfer (automatic),64.45,1867.6,DSL,...,Yes,Yes,No,No,Female,1,No,No,Yes,0
1,2044,6557-BZXLQ,2018-10-01,NaT,Month-to-month,No,Electronic check,69.65,1043.3,Fiber optic,...,No,No,No,No,Male,1,No,No,No,0


In [2]:
# разделение выборки
from sklearn.model_selection import train_test_split

features = ["monthly_charges", "total_charges", "senior_citizen"]
target = "target"

split_column = "customer_id"  # Замените на подходящий столбец
stratify_column = "stratify_column"  # Зафиксируйте целевую переменную для стратификации
test_size = 0.2  # Размер тестовой выборки

df = df.sort_values(by=[split_column])

X_train, X_test, y_train, y_test = train_test_split(
    df[features], df[target], test_size=test_size, shuffle=False
)

print(f"Размер выборки для обучения: {X_train.shape}")
print(f"Размер выборки для теста: {X_test.shape}")


Размер выборки для обучения: (5634, 3)
Размер выборки для теста: (1409, 3)


In [3]:
import os
import optuna
from optuna.samplers import TPESampler
from optuna.integration.mlflow import MLflowCallback
from catboost import CatBoostClassifier
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, precision_score, recall_score, f1_score, log_loss
from collections import defaultdict
import mlflow
import numpy as np


os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net"
os.environ["AWS_ACCESS_KEY_ID"] = "YCAJE3Nlz8iDILW5VTYM1ihQB"
os.environ["AWS_SECRET_ACCESS_KEY"] = "YCPjvS7uwhvJpUj3bKm8X-IX4QAwBIVsvX61IL44"

mlflow.set_tracking_uri(f"http://127.0.0.1:5000")
mlflow.set_registry_uri(f"http://127.0.0.1:5000")

EXPERIMENT_NAME = "churn"
RUN_NAME = "model_bayesian_search"
MLFLOW_PARENT_RUN_ID = "0e60e1de414a4a348318454d38885e0b"

STUDY_DB_NAME = "sqlite:///local.study.db"
STUDY_NAME = "churn_model"


def objective(trial: optuna.Trial) -> float:
    param = {
      "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.1, log=True),
      "depth": trial.suggest_int("depth", 1, 12),
      "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 0.1, 5),
      "random_strength": trial.suggest_float("random_strength", 0.1, 5),
      "loss_function": "Logloss",
      "task_type": "CPU",
      "random_seed": 0,
      "iterations": 300,
      "verbose": False,
    }
    model = CatBoostClassifier(**param)

    skf = StratifiedKFold(n_splits=2)
    metrics = defaultdict(list)
    for i, (train_index, val_index) in enumerate(skf.split(X_train, y_train)):
        train_x = X_train.iloc[train_index]
        train_y = y_train.iloc[train_index]
        val_x = X_train.iloc[val_index]
        val_y = y_train.iloc[val_index]

        model.fit(train_x, train_y)
        prediction = model.predict(val_x)
        probas = model.predict_proba(val_x)[:, 1]

        _, err1, _, err2 = confusion_matrix(val_y, prediction, normalize='all').ravel()
        auc = roc_auc_score(val_y, probas)
        precision = precision_score(val_y, prediction, average='weighted')
        recall = recall_score(val_y, prediction, average='weighted')
        f1 = f1_score(val_y, prediction, average='weighted')
        logloss = log_loss(val_y, probas)
        
        metrics["err1"].append(err1)
        metrics["err2"].append(err2)
        metrics["auc"].append(auc)
        metrics["precision"].append(precision)
        metrics["recall"].append(recall)
        metrics["f1"].append(f1)
        metrics["logloss"].append(logloss)


    # ваш код здесь #
    auc = np.mean(metrics["auc"])
		

    return auc


experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if not experiment:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
else:
    experiment_id = experiment.experiment_id
    

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    existing_run_id = run_id

    mlflc = MLflowCallback(
        tracking_uri=f"http://127.0.0.1:5000", 
        metric_name="AUC", 
        create_experiment=False,
        mlflow_kwargs={'experiment_id': experiment_id, 'tags': {'run_id': run_id}, 'nested':True},
    )

    study = optuna.create_study(
        sampler=optuna.samplers.TPESampler(),
        direction="maximize", 
        study_name=STUDY_NAME, 
        storage=STUDY_DB_NAME, 
        load_if_exists=True
    )
    study.optimize(objective, n_trials=10, callbacks=[mlflc])
    best_params = study.best_params # ваш код здесь #

print(f"Number of finished trials: {len(study.trials)}")
print(f"Best params: {best_params}")

/home/mle-user/mle-projects/mle-mlflow/.venv_notebook/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_108547/1146942977.py:89: ExperimentalWarning: MLflowCallback is experimental (supported from v1.4.0). The interface can change in the future.
  mlflc = MLflowCallback(
[I 2025-09-16 06:55:12,457] Using an existing study with name 'churn_model' instead of creating a new one.
[I 2025-09-16 06:55:13,463] Trial 25 finished with value: 0.8099234892139668 and parameters: {'learning_rate': 0.009611997132010035, 'depth': 3, 'l2_leaf_reg': 2.614047064397579, 'random_strength': 2.0000010963260806}. Best is trial 23 with value: 0.815266173936608.
[I 2025-09-16 06:55:14,376] Trial 26 finished with value: 0.8121738816425244 and parameters: {'learning_rate': 0.044481652218317756, 'depth': 4, 'l2_leaf_r

Number of finished trials: 35
Best params: {'learning_rate': 0.022500019511261276, 'depth': 4, 'l2_leaf_reg': 2.176483697162663, 'random_strength': 2.669479486895823}


In [4]:
# проверка модели с лучшими параметрами

model_best = CatBoostClassifier(
    **best_params, verbose=False
)

model_best.fit(X_train, y_train)

prediction = model_best.predict(X_test)
probas = model_best.predict_proba(X_test)[:, 1]

In [5]:
from sklearn.metrics import confusion_matrix, roc_auc_score, precision_score, recall_score, f1_score, log_loss

metrics = {}

_, err1, _, err2 = confusion_matrix(y_test, prediction, normalize='all').ravel()
auc = roc_auc_score(y_test, probas)
precision = precision_score(y_test, prediction)
recall = recall_score(y_test, prediction)
f1 = f1_score(y_test, prediction)
logloss = log_loss(y_test, prediction)

# сохранение метрик в словарь
metrics["err1"] = err1
metrics["err2"] = err2
metrics["auc"] = auc
metrics["precision"] = precision
metrics["recall"] = recall
metrics["f1"] = f1
metrics["logloss"] = logloss

In [13]:
# логируем результат
import mlflow

EXPERIMENT_NAME = "churn"
RUN_NAME = 'model_bayesian_search' # ваш код здесь
REGISTRY_MODEL_NAME = "churn_model_arvas"

os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net"
os.environ["AWS_ACCESS_KEY_ID"] = "YCAJE3Nlz8iDILW5VTYM1ihQB"
os.environ["AWS_SECRET_ACCESS_KEY"] = "YCPjvS7uwhvJpUj3bKm8X-IX4QAwBIVsvX61IL44"

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_registry_uri("http://localhost:5000")

# настройки для логирования в MLFlow
pip_requirements = 'requirements.txt'
signature = mlflow.models.infer_signature(X_test, prediction)
input_example = X_test[:10]

experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id


with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id, run_id=existing_run_id) as run:
    run_id = run.info.run_id
    mlflow.log_params(best_params)
    mlflow.log_metrics(metrics)
    model_info = mlflow.catboost.log_model(
        cb_model=model_best,
        artifact_path='cv',
        input_example=input_example,
        signature=signature,
        registered_model_name=REGISTRY_MODEL_NAME,
        pip_requirements=pip_requirements
    )
    # Для сохранения в файл
    model_file_path = "model.pkl"
    with open(model_file_path, "wb") as f:
        pickle.dump(model_best, f)  # Сохранить модель в файл model.pkl
    
    # Логирование файла
    mlflow.log_artifact(model_file_path, artifact_path="cv")


/home/mle-user/mle-projects/mle-mlflow/.venv_notebook/lib/python3.10/site-packages/mlflow/models/signature.py:212: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  inputs = _infer_schema(model_input) if model_input is not None else None
2025/09/16 06:59:06 WARNING mlflow.models.model: Logging model metadata to the tracking server has failed. The model artifacts have been logge